# Anomaly detection with PyOD\n\nNotebook version of [AnomalyDetectionUsingPyOD](https://github.com/JaminJeong/AnomalyDetectionUsingPyOD) (daily minimum temperatures, sliding windows, multiple detectors). Temperatures are loaded with **`%%cribl_search`** using Cribl Search [`externaldata`](https://docs.cribl.io/search/externaldata/) (same public CSV as the upstream example—see [daily-min-temperatures.csv](https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv) and [Machine Learning Mastery](https://machinelearningmastery.com/time-series-datasets-for-machine-learning/)). [PyOD](https://github.com/yzhao062/pyod) is installed in **Setup** via `micropip`.

## How to run\n\n1. Run **Setup** once (downloads PyOD stack; first run can take several minutes).\n2. Run **Shared preprocessing** (Python helpers and constants).\n3. Run **Load temperatures** (`%%cribl_search` with `externaldata`) — requires this notebook to run **inside a hosted Cribl app** where Search is available (like other `%%cribl_search` examples).\n4. Run **Materialize dataframe**, then **train/test windows**, then each **detector** cell (each draws its own figure). **Variational Auto Encoder** and **Auto Encoder** often **do not run** in Pyodide. The upstream **LOCI** screenshots are not part of the upstream script's active models; this notebook keeps all **18** detectors from `anomaly_detection_all_model.py`.

### Setup\n\nInstalls **SciPy**, **scikit-learn**, and **joblib**, then **PyOD** with `deps=False` so micropip does not pull **Numba** (declared by PyPI metadata but unavailable on Pyodide). Requires network access to PyPI/jsDelivr (allow-listed in the app `proxies.yml`).

In [ ]:
import micropip\n\n# PyPI declares `numba` for pyod, but Numba has no Pyodide wheel — install the\n# scientific stack explicitly, then pyod without transitive deps.\nawait micropip.install(['scipy', 'scikit-learn', 'joblib'])\nawait micropip.install(['pyod'], deps=False)

### Shared preprocessing\n\nImports, constants (`window_size`, `contamination`), helper functions for sliding windows and plots, and the LOF list used by **LSCP**. Run **Setup** above first so `pyod` is available.

In [ ]:
import numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom pyod.models.lof import LOF\n\nwindow_size = 30\ncontamination = 0.25\noutliers_fraction = contamination\nrandom_state = 42\n\n\ndef make_data_sampling(data, window_size):\n    n_samples = len(data)\n    if n_samples < window_size:\n        raise ValueError('window size is too long for series')\n    return np.array([data[i : i + window_size] for i in range(n_samples - window_size + 1)])\n\n\ndef plot_anomalies(title, real_value, y_pred):\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(14, 3.5))\n    ax.plot(np.arange(len(real_value)), real_value, label='real')\n    scatter_x = [i for i, lab in enumerate(y_pred) if lab == 1]\n    scatter_y = [real_value[i] for i in scatter_x]\n    ax.scatter(scatter_x, scatter_y, color='0.5', edgecolor='r', label='anomaly')\n    ax.set_title(title)\n    ax.legend(loc='upper right')\n    plt.tight_layout()\n    plt.show()\n    plt.close(fig)\n\n\ndef run_detector(title, clf):\n    clf.fit(df_data_train)\n    y_pred = clf.predict(df_data_test)\n    real_value = df_data_test[:, -1]\n    plot_anomalies(title, real_value, y_pred)\n

### Load temperatures (`externaldata`)\n\nFetches the CSV through **[`externaldata`](https://docs.cribl.io/search/externaldata/)** so retrieval happens in **Cribl Search**, not via `pd.read_csv` in Pyodide. Uses stock datatype **`generic_csv`**. If Search rejects the datatype on your tenant, try **`CSV`** (v1 stock) or your admin’s comma-separated datatype.

In [ ]:
%%cribl_search var=temp_df lang=kql limit=0 preview=false earliest=-50y latest=now\nexternaldata\n[\n  "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"\n]\nwith(\n  datatype="generic_csv"\n)

In [ ]:
# Materialize `df_data` from Search results (columns follow the CSV header).\ndf_data = temp_df.copy()\n_norm = {str(c).strip().lower(): c for c in df_data.columns}\nfor std in ('Date', 'Temp'):\n    lo = std.lower()\n    if std not in df_data.columns and lo in _norm:\n        df_data = df_data.rename(columns={_norm[lo]: std})\ndf_data['Temp'] = pd.to_numeric(df_data['Temp'], errors='coerce')\ndf_data = df_data.dropna(subset=['Temp']).reset_index(drop=True)\ndf_data.head()

In [ ]:
raw_train, raw_test = train_test_split(\n    np.array(df_data['Temp']), test_size=0.2, shuffle=False\n)\n\ndetector_list = [\n    LOF(n_neighbors=n) for n in (5, 10, 15, 20, 25, 30, 35, 40, 45, 50)\n]\n\ndf_data_train = make_data_sampling(raw_train, window_size)\ndf_data_test = make_data_sampling(raw_test, window_size)\nprint('train windows', df_data_train.shape, 'test windows', df_data_test.shape)

### Train series (subsampled)\n\nSame idea as `GenGraph('train')` in the upstream `save_graph.py`.

In [ ]:
import matplotlib.pyplot as plt\n\nfig, ax = plt.subplots(figsize=(14, 3))\nax.plot(raw_train[::30])\nax.set_title('Train temperatures (every 30th point)')\nplt.tight_layout()\nplt.show()\nplt.close(fig)

### Test series (subsampled)\n\nSame idea as `GenGraph('test')`.

In [ ]:
import matplotlib.pyplot as plt\n\nfig, ax = plt.subplots(figsize=(14, 3))\nax.plot(raw_test[::30])\nax.set_title('Test temperatures (every 30th point)')\nplt.tight_layout()\nplt.show()\nplt.close(fig)

## Detectors\n\nEach section fits **one** model on `df_data_train`, scores `df_data_test`, and plots the **last timestep** of each window against anomalies (matching `BasicGenGraph` in the upstream project).

### Angle-based Outlier Detector (ABOD)

In [ ]:
from pyod.models.abod import ABOD\n\ntitle = 'Angle-based Outlier Detector (ABOD)'\ntry:\n    clf = ABOD(contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Cluster-based Local Outlier Factor (CBLOF)

In [ ]:
from pyod.models.cblof import CBLOF\n\ntitle = 'Cluster-based Local Outlier Factor (CBLOF)'\ntry:\n    clf = CBLOF(contamination=outliers_fraction, check_estimator=False, random_state=random_state, n_clusters=15)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Feature Bagging

In [ ]:
from pyod.models.feature_bagging import FeatureBagging\n\ntitle = 'Feature Bagging'\ntry:\n    clf = FeatureBagging(LOF(n_neighbors=35), contamination=outliers_fraction, random_state=random_state)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Histogram-base Outlier Detection (HBOS)

In [ ]:
from pyod.models.hbos import HBOS\n\ntitle = 'Histogram-base Outlier Detection (HBOS)'\ntry:\n    clf = HBOS(contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Isolation Forest

In [ ]:
from pyod.models.iforest import IForest\n\ntitle = 'Isolation Forest'\ntry:\n    clf = IForest(contamination=outliers_fraction, random_state=random_state)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### K Nearest Neighbors (KNN)

In [ ]:
from pyod.models.knn import KNN\n\ntitle = 'K Nearest Neighbors (KNN)'\ntry:\n    clf = KNN(contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Average KNN

In [ ]:
from pyod.models.knn import KNN\n\ntitle = 'Average KNN'\ntry:\n    clf = KNN(method='mean', contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Median KNN

In [ ]:
from pyod.models.knn import KNN\n\ntitle = 'Median KNN'\ntry:\n    clf = KNN(method='median', contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Local Outlier Factor (LOF)

In [ ]:
from pyod.models.lof import LOF\n\ntitle = 'Local Outlier Factor (LOF)'\ntry:\n    clf = LOF(n_neighbors=35, contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Minimum Covariance Determinant (MCD)

In [ ]:
from pyod.models.mcd import MCD\n\ntitle = 'Minimum Covariance Determinant (MCD)'\ntry:\n    clf = MCD(contamination=outliers_fraction, random_state=random_state)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### One-class SVM (OCSVM)

In [ ]:
from pyod.models.ocsvm import OCSVM\n\ntitle = 'One-class SVM (OCSVM)'\ntry:\n    clf = OCSVM(contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Principal Component Analysis (PCA)

In [ ]:
from pyod.models.pca import PCA\n\ntitle = 'Principal Component Analysis (PCA)'\ntry:\n    clf = PCA(contamination=outliers_fraction, random_state=random_state)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Stochastic Outlier Selection (SOS)

In [ ]:
from pyod.models.sos import SOS\n\ntitle = 'Stochastic Outlier Selection (SOS)'\ntry:\n    clf = SOS(contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Locally Selective Combination (LSCP)

In [ ]:
from pyod.models.lscp import LSCP\n\ntitle = 'Locally Selective Combination (LSCP)'\ntry:\n    clf = LSCP(detector_list, contamination=outliers_fraction, random_state=random_state)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Connectivity-Based Outlier Factor (COF)

In [ ]:
from pyod.models.cof import COF\n\ntitle = 'Connectivity-Based Outlier Factor (COF)'\ntry:\n    clf = COF(n_neighbors=35, contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Subspace Outlier Detection (SOD)

In [ ]:
from pyod.models.sod import SOD\n\ntitle = 'Subspace Outlier Detection (SOD)'\ntry:\n    clf = SOD(contamination=outliers_fraction)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Variational Auto Encoder

In [ ]:
from pyod.models.vae import VAE\n\ntitle = 'Variational Auto Encoder'\ntry:\n    clf = VAE(epochs=15, contamination=contamination, gamma=0.8, capacity=0.2, batch_size=window_size)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Auto Encoder

In [ ]:
from pyod.models.auto_encoder import AutoEncoder\n\ntitle = 'Auto Encoder'\ntry:\n    clf = AutoEncoder(epochs=15, contamination=contamination, batch_size=window_size)\n    run_detector(title, clf)\nexcept Exception as err:\n    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

## Next steps\n\nTune `window_size` and `contamination`, try your own series, or export scores with `clf.decision_function(df_data_test)` once a model fits.